<div style="display: flex; align-items: center;">
  <img src="../Images/Logos/DecisionIntelligence-DecisionFraming-Logo.png" width="60px" style="margin-right: 10px;">
  <span style="font-size: 1.5em; font-weight: bold;">Decision Framing - Six Thinking Hats</span>
</div>

Decision Intelligence applied in this module:

* Using decision frames to evaluate a major purchase from multiple perspectives.
* Applying Edward de Bono's Six Thinking Hats as reusable decision lenses.
* Asking GenAI to prioritize the most useful frames for a high-stakes decision.
* Combining prompt design and Microsoft.Extensions.AI to produce strict Markdown-table outputs.

-----
### Step 1 - Initialize Configuration Builder & Build the AI Extensions Responses API Orchestration

In [2]:
#r "nuget: Microsoft.Extensions.Configuration, 10.0.12"
#r "nuget: Microsoft.Extensions.Configuration.Json, 10.0.12"
#r "nuget: System.Text.Json, 10.0.12"

using Microsoft.Extensions.Configuration.Json;
using Microsoft.Extensions.Configuration;
using System.IO;
using System;

var configurationBuilder = new ConfigurationBuilder()
    .SetBasePath(Directory.GetCurrentDirectory())
    .AddJsonFile("local.settings.json", optional: true, reloadOnChange: true)
    .AddJsonFile("secrets.settings.json", optional: true, reloadOnChange: true);
var config = configurationBuilder.Build();

// IMPORTANT: You ONLY NEED either Azure OpenAI or OpenAI connection info, not both.
// Azure OpenAI Connection Info
var azureOpenAIEndpoint = config["AzureOpenAI:Endpoint"];
var azureOpenAIAPIKey = config["AzureOpenAI:APIKey"];
var azureOpenAIModelDeploymentName = config["AzureOpenAI:ModelDeploymentName"];
// OpenAI Connection Info 
var openAIAPIKey = config["OpenAI:APIKey"];
var openAIModelId = config["OpenAI:ModelId"];
// AzureOpenAI or OpenAI selection flag
var useAzureOpenAI = bool.Parse(config["AzureOpenAI:UseAzureOpenAI"] ?? "true");

Installed Packages Microsoft.Extensions.Configuration, 10.0.12 Microsoft.Extensions.Configuration.Json, 10.0.12 System.Text.Json, 10.0.12

In [3]:
// Import the Microsoft Extensions AI NuGet Packages
#r "nuget: Microsoft.Extensions.AI, 10.10.0"
#r "nuget: Microsoft.Extensions.AI.Abstractions, 10.10.0"
#r "nuget: Microsoft.Extensions.AI.OpenAI, 10.10.0"

// Import Azure & OpenAI NuGet Packages
#r "nuget: Azure.Identity, 1.21.0"
#r "nuget: OpenAI, 2.13.0"

using Azure;
using Microsoft.Extensions.AI;
using Microsoft.Extensions.Configuration;
using OpenAI;
using System.ClientModel;
using System.ComponentModel;
using System.Text.Json;


// Create the IChatClient based on the selected service
IChatClient chatClient;

// Create a new MEAI ChatClient instance
if (useAzureOpenAI)
{
    Console.WriteLine("Using Azure OpenAI Service");

    var apiKeyCredential = new ApiKeyCredential(azureOpenAIAPIKey!);

    var azureOpenAIClient = new OpenAIClient(
        apiKeyCredential,
        new OpenAIClientOptions
        {
            Endpoint = new Uri($"{azureOpenAIEndpoint!.TrimEnd('/')}/openai/v1")
        });

    #pragma warning disable OPENAI001
    var responsesClient = azureOpenAIClient.GetResponsesClient();
    chatClient = responsesClient.AsIChatClient(azureOpenAIModelDeploymentName);
}
else
{
    Console.WriteLine("Using OpenAI Service");

    var apiKeyCredential = new ApiKeyCredential(azureOpenAIAPIKey);
    var openAIClient = new OpenAIClient(apiKeyCredential);

    #pragma warning disable OPENAI001
    var responsesClient = openAIClient.GetResponsesClient();
    chatClient = responsesClient.AsIChatClient(defaultModelId: openAIModelId);
}

#pragma warning restore OPENAI001

var decisionFrameSystemPrompt = """
You are a Decision Intelligence assistant.
Help the user explore options, evaluate tradeoffs, reason through uncertainty, solve problems,
and apply systems thinking to personal, professional, strategic, and operational decisions.

Provide responses that are structured, logical, balanced, analytical, and pragmatic.
Improve the user's judgment rather than simply making the choice for them.

Output Format Instructions:
Return only a valid Markdown table.
Do not include prose before or after the table.
Do not wrap the table in triple backticks.
Do not use Markdown headings anywhere in the response.
Do not use #, ##, ###, ####, or any heading syntax inside or outside table cells.
Do not use horizontal rules of any kind.
Do not use ---, ***, or ___ anywhere in the response.
Do not use bullet lists or numbered lists inside table cells.
Use plain text column headers only.
If line breaks are needed inside a cell, use <br>.
Escape any literal pipe characters inside cell content so the table structure is not broken.
Make the table syntactically valid Markdown with exactly one header row and one separator row.
""";

async Task<string> RunDecisionFramePromptAsync(string userPrompt, ChatOptions options = null)
{
    var chatMessages = new List<ChatMessage>
    {
        new(ChatRole.System, decisionFrameSystemPrompt),
        new(ChatRole.User, userPrompt)
    };

    ChatResponse response = await chatClient.GetResponseAsync(chatMessages, options);
    return response.Text ?? string.Empty;
}

#pragma warning disable OPENAI001

var decisionFrameChatOptions = new ChatOptions
{
    RawRepresentationFactory = _ => new OpenAI.Responses.CreateResponseOptions
    {
        Model = useAzureOpenAI ? azureOpenAIModelDeploymentName : openAIModelId,
        ReasoningOptions = new OpenAI.Responses.ResponseReasoningOptions
        {
            ReasoningEffortLevel = OpenAI.Responses.ResponseReasoningEffortLevel.Medium,
            ReasoningSummaryVerbosity = OpenAI.Responses.ResponseReasoningSummaryVerbosity.Detailed
        },
        StoredOutputEnabled = false
    }
};

#pragma warning restore OPENAI001

Installed Packages Azure.Identity, 1.21.0 Microsoft.Extensions.AI, 10.10.0 Microsoft.Extensions.AI.Abstractions, 10.10.0 Microsoft.Extensions.AI.OpenAI, 10.10.0 OpenAI, 2.13.0

Using Azure OpenAI Service


-----
### Step 2 - Decision Frames (Thinking Hats) Scenario for a Major Purchase

> 📜 **_"The mere formulation of a problem is far more essential than its solution, which may be merely a matter of mathematical or experimental skill. To raise new questions, new possibilities, to regard old problems from a new angle requires creative imagination and marks real advances in science."_**
>
> -- <cite>Albert Einstein (Theoretical Physicist, Best known for developing the theory of relativity)</cite>  

The code below lists potential decision frames using the Six Thinking Hats framework for purchasing a house. Generative AI can help apply each hat as a distinct decision lens, making it easier to inspect the same choice from multiple useful perspectives.

In [4]:
// Decision Frames Prompt that lists the Six Thinking Hats approach to purchasing a house.
var decisionFramesPrompt = """
You are planning on purchasing a new home.
Use Edward de Bono's Six Thinking Hats to frame the decision-making process.

Instructions:
Create one row for each of the six thinking hats.
For each row, explain how that decision frame should guide the house-purchase decision.
Include practical questions and evidence the buyer should gather.

Output Requirements:
Return one Markdown table with these columns:
Decision Frame | Focus Area | House-Purchase Questions | Evidence To Gather | Decision Value
""";

var decisionFramesResponseText = await RunDecisionFramePromptAsync(
    decisionFramesPrompt,
    decisionFrameChatOptions);

decisionFramesResponseText.DisplayAs("text/markdown");

| Decision Frame | Focus Area | House-Purchase Questions | Evidence To Gather | Decision Value |
|---|---|---|---|---|
| White Hat: Facts and Information | Objective facts, affordability, property condition, and market data | What is the total monthly cost including mortgage, taxes, insurance, utilities, maintenance, and fees?<br>What are the property's size, age, condition, and legal restrictions?<br>What comparable homes have recently sold nearby? | Mortgage preapproval and rate estimates<br>Verified income, savings, debts, and cash reserves<br>Inspection report, seller disclosures, title report, survey, tax records, utility costs, homeowners association documents, and recent comparable sales | Establishes a factual baseline and prevents decisions based on incomplete or inaccurate information |
| Red Hat: Feelings and Intuition | Emotional response, comfort, identity, and unspoken concerns | How do I feel when entering and living in the home?<br>Do I feel safe, comfortable, and able to picture daily life there?<br>Am I excited about this home, or am I reacting mainly to pressure, scarcity, or fear of missing out? | Written reactions after each viewing<br>Separate ratings for comfort, location, layout, and emotional fit<br>Input from household members, while distinguishing their preferences from social pressure | Surfaces meaningful preferences and warning signals that may not appear in financial analysis |
| Black Hat: Caution and Risk | Downside scenarios, constraints, hidden costs, and potential regret | What could make this purchase financially or practically damaging?<br>Could I afford it if income fell, rates rose, or major repairs were needed?<br>Are there flood, fire, environmental, legal, neighborhood, resale, or maintenance risks?<br>What assumptions would have to be wrong for this decision to fail? | Stress tests using higher interest rates and reduced income<br>Professional inspection and specialist reports where needed<br>Insurance quotes, flood and hazard maps, planning applications, crime data, building permits, repair estimates, and resale analysis | Identifies preventable risks, tests resilience, and reduces the chance of expensive surprises |
| Yellow Hat: Benefits and Value | Advantages, opportunities, long-term usefulness, and upside | What benefits does ownership provide compared with renting or buying alternatives?<br>Will the location, layout, energy efficiency, and transport access support future needs?<br>Could improvements increase enjoyment or value?<br>What would make this home a strong long-term fit? | Comparison with alternative properties and renting costs<br>Commute and school information<br>Expected maintenance and energy costs<br>Neighborhood development plans, renovation feasibility, and estimated future demand | Clarifies the purchase's positive case and distinguishes durable value from optimistic assumptions |
| Green Hat: Alternatives and Creativity | Options, negotiation, adaptations, and nonstandard solutions | Could I buy a different property, wait, rent longer, or change the target area?<br>Could the home work through renovation, a different financing structure, or a negotiated price?<br>What features are essential, desirable, or replaceable?<br>Are there creative ways to reduce risk or cost? | Shortlist of alternative homes and locations<br>Renovation concepts with contractor estimates<br>Multiple loan scenarios<br>Seller concessions, price comparisons, flexible completion options, and rental-versus-buy calculations | Prevents premature commitment and expands the solution set beyond accepting or rejecting one property |
| Blue Hat: Process and Decision Control | Decision criteria, sequencing, governance, and final review | What are the must-have criteria and deal-breakers?<br>Which facts remain unknown, and who will verify them?<br>What conditions must be satisfied before making an offer or completing the purchase?<br>How will I avoid anchoring, urgency, and confirmation bias?<br>What is the exit point if evidence changes? | Written decision scorecard with weighted criteria<br>Budget ceiling and walk-away price<br>Due diligence checklist, independent professional opinions, cooling-off or contingency terms, decision deadline, and documented assumptions | Creates a disciplined process, integrates the other perspectives, and ensures the final choice remains aligned with priorities and risk tolerance |

This can be enhanced further by asking GenAI to suggest which decision frames might be most useful for a high-stakes decision. The prompt below lists all of the decision frames, prioritizes them by expected impact and influence, and recommends which frames deserve the most attention.

This lets the AI automation help optimize the decision process while the human decision maker remains responsible for the actual decision selection and execution.

In [4]:
// Decision Frames Prompt that ranks the Six Thinking Hats approach to purchasing a house.
var decisionFramesListPrompt = """
You are planning on purchasing a new home.
Use Edward de Bono's Six Thinking Hats to frame the decision-making process.

Instructions:
List the six thinking hats in the order of expected impact and influence from most to least for a house-purchase decision.
For each thinking hat, explain why it has that ranking.
Recommend whether the buyer should focus on that hat heavily, moderately, or lightly.

Output Requirements:
Return one Markdown table with these columns:
Rank | Decision Frame | Why It Matters For A House Purchase | Impact / Influence | Recommended Focus
""";

var decisionFramesListResponseText = await RunDecisionFramePromptAsync(
    decisionFramesListPrompt,
    decisionFrameChatOptions);

decisionFramesListResponseText.DisplayAs("text/markdown");

| Rank | Decision Frame | Why It Matters For A House Purchase | Impact / Influence | Recommended Focus |
|---|---|---|---|---|
| 1 | White Hat: Facts and Information | Establishes affordability, income stability, mortgage rates, taxes, insurance, maintenance costs, property condition, location data, and comparable sales. Reliable facts determine whether the purchase is financially and practically viable. | Very high: factual errors can create unaffordable payments, overpayment, or unexpected ownership costs. | Heavily |
| 2 | Black Hat: Risks and Caution | Tests downside scenarios such as job loss, interest-rate changes, falling property values, structural defects, poor resale prospects, restrictive covenants, and excessive repair needs. | Very high: identifying risks before committing can prevent major financial and lifestyle harm. | Heavily |
| 3 | Blue Hat: Process and Control | Provides a disciplined decision process, including defining requirements, setting a budget ceiling, comparing properties consistently, completing due diligence, and establishing conditions for walking away. | High: strong process reduces emotional overcommitment, missed information, and impulsive decisions. | Heavily |
| 4 | Red Hat: Emotions and Intuition | Captures feelings about safety, belonging, neighborhood fit, commute comfort, and whether the home feels suitable. These reactions matter because the buyer will live with the decision daily. | Moderate to high: emotional fit influences satisfaction, but strong feelings can also cause buyers to ignore risks or exceed their budget. | Moderately |
| 5 | Yellow Hat: Benefits and Value | Identifies potential benefits such as long-term stability, lifestyle improvement, location advantages, renovation upside, community quality, and possible equity growth. | Moderate: benefits help compare acceptable options, but optimistic assumptions should not override affordability or risk analysis. | Moderately |
| 6 | Green Hat: Alternatives and Creativity | Encourages considering different neighborhoods, property types, financing structures, renovation strategies, timing, co-buying arrangements, or waiting before purchasing. | Moderate to low for evaluating one specific home, but valuable when the current option does not fit the buyer's constraints. | Lightly, increasing to moderately when no option clearly meets the requirements |

-----
### Step 3 - Applying the Decision Frame to a Decision 

So far, Generative AI has helped create decision frames with the Six Thinking Hats framework, rank them from most to least impactful, and identify where to focus attention. The next step is to apply the selected decision frames directly to the house-purchase decision.

This notebook uses a single optimized prompt for that final step: the model performs the ranking and frame selection internally, then returns only the applied decision-frame approach as a Markdown table.

In [5]:
// Decision Frames Prompt that internally ranks and selects the strongest frames, then only returns the applied approach.
var optimizedDecisionFramesPrompt = """
You are planning on purchasing a new home.
Use Edward de Bono's Six Thinking Hats to frame the decision-making process.

Internal Instructions:
Internally identify an approach for each of the six thinking hats.
Internally rank the thinking hats by expected impact and influence for the house-purchase decision.
Internally select the most useful decision frames to focus on.
Do not show the internal ranking, hidden selection notes, or hidden reasoning steps.

Visible Output Instructions:
From the internally selected decision frames, create an applied approach for purchasing a house.
The response should help the buyer know what to inspect, what questions to ask, what evidence to gather, and what action to take.

Output Requirements:
Return one Markdown table with these columns:
Selected Decision Frame | Applied House-Purchase Approach | Key Questions | Evidence / Signals | Buyer Action
""";

var optimizedDecisionFramesResponseText = await RunDecisionFramePromptAsync(
    optimizedDecisionFramesPrompt,
    decisionFrameChatOptions);

optimizedDecisionFramesResponseText.DisplayAs("text/markdown");

| Selected Decision Frame | Applied House-Purchase Approach | Key Questions | Evidence / Signals | Buyer Action |
|---|---|---|---|---|
| Blue Hat: Process and governance | Define the decision criteria, sequence the work, assign deadlines, and set a walk-away rule before becoming emotionally committed. | What must the home satisfy? What is the maximum all-in monthly cost? Who has authority to decide? What conditions must be met before making an offer? | Written priorities, affordability ceiling, inspection and financing deadlines, agreed deal-breakers | Create a decision checklist and timeline. Do not make an offer until financing, inspections, comparable sales, and legal review are planned. |
| White Hat: Facts and evidence | Build an objective picture of the property, neighborhood, ownership costs, and financing rather than relying on presentation or assumptions. | What are comparable sale prices? What are taxes, insurance, utilities, maintenance, and association fees? How long has the property been listed? What permits, disclosures, and repair records exist? | Preapproval letter, lender estimates, tax records, title report, seller disclosures, inspection report, permit history, comparable sales, insurance quote | Verify important claims independently. Calculate total monthly and five-year costs using conservative assumptions. |
| Red Hat: Feelings and intuition | Notice emotional reactions without allowing them to substitute for evidence. Identify whether excitement, fear, urgency, or attachment is influencing judgment. | What specifically creates confidence or discomfort? Would the decision feel different without staging or pressure? Are we overlooking concerns because we love the location or design? | Strong emotional swings, fear of missing out, pressure from the seller or agent, repeated rationalization of concerns | Record intuitive concerns, then test each against facts. Use a cooling-off period where possible and avoid bidding solely to avoid losing. |
| Black Hat: Risks and failure modes | Stress-test the purchase for financial, physical, legal, environmental, resale, and lifestyle risks. Assume that optimistic assumptions may be wrong. | What could make this unaffordable? What defects may be hidden? Is the area vulnerable to flooding, fire, noise, or declining services? What happens if income falls or repairs arrive early? | Inspection findings, structural or moisture issues, insurance exclusions, flood and hazard maps, high maintenance costs, unresolved permits, restrictive covenants, weak resale demand | Price the risk, negotiate credits or repairs, obtain specialist inspections, and walk away if risks are uninsurable, unaffordable, or inadequately understood. |
| Yellow Hat: Benefits and opportunity | Evaluate the durable value the home could provide, including location, functionality, flexibility, quality of life, and likely resilience. | Does it support current and future needs? What benefits are difficult or costly to replicate elsewhere? Is the location likely to remain desirable? Could improvements add meaningful value? | Commute and service access, school or community fit, adaptable layout, condition relative to price, neighborhood investment, realistic renovation estimates | Compare benefits against alternatives and pay for them only when they are durable, evidenced, and within the affordability limit. |
| Green Hat: Alternatives and creative options | Expand the choice set before deciding: different properties, locations, financing structures, offer terms, renovation plans, or delaying the purchase. | What other homes meet the essential criteria? Could renting and saving be superior? Could a lower price offset renovation needs? Would a different closing date or contingency improve the deal? | Shortlist comparison, rent-versus-buy analysis, renovation quotes, alternative loan scenarios, opportunity cost of the down payment | Generate at least three credible alternatives, compare them using the same criteria, and make the purchase only if this home remains the strongest option after alternatives are considered. |